# Lesson 7b: Sequence Models — Practical

7a derived the RNN recurrence and backpropagation through time from
scratch, and derived why LSTM gates preserve gradient magnitude that a
plain RNN's repeated $\tanh'(\cdot) W_{hh}$ product does not. This
notebook builds the production version — `nn.LSTM` — trains it as a real
character-level language model with minibatches, teacher forcing and
gradient clipping, reports the standard language-modelling metric
(**perplexity**) on held-out text, and samples from it at several
temperatures to show what that one number actually buys in practice.

By the end of this notebook you will have:
- trained an **`nn.LSTM`-based character language model** with
  minibatched, gradient-clipped optimisation,
- measured **training loss and held-out perplexity** across training and
  related the plateau to what 7a's gradient argument predicts an LSTM
  should be able to do that a plain RNN struggles with,
- demonstrated **teacher forcing** during training versus **autoregressive
  sampling** at generation time, and
- generated samples at several **temperatures** and characterised how
  each changes the text's coherence and diversity.

## Introduction

7a trained a from-scratch, 64-unit vanilla RNN on a single 775-character
paragraph purely to verify that its hand-derived forward and backward
pass actually optimised something. This notebook trains a properly
sized LSTM on a larger held-out-validated corpus and reports the metric
every language model is actually judged on: **perplexity**, the
exponentiated average cross-entropy per token — intuitively, "the
model's ground-truth next token felt about this equally likely among $N$
options," so a perplexity near the vocabulary size is a model that has
learned nothing, and small values mean the true next character is
reliably one of a handful of good guesses.

## Setup

In [ ]:
# Fixed seeds: every stochastic step (weight init, minibatch sampling,
# sampling temperature draws) is reproducible.
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (6, 4)
device = torch.device("cpu")
print("numpy:", np.__version__)
print("torch:", torch.__version__)

### A tiny text corpus, with a held-out split

Same source as 7a (the public-domain opening of *Pride and Prejudice*,
embedded directly — no download), extended further into the chapter so
there is enough text for a genuine train/validation split. The last 15%
of the text is held out and never trained on; perplexity is reported on
that held-out slice only.

In [ ]:
paragraph_1 = [
    "It is a truth universally acknowledged, that a single man in possession ",
    "of a good fortune, must be in want of a wife. However little known the ",
    "feelings or views of such a man may be on his first entering a ",
    "neighbourhood, this truth is so well fixed in the minds of the ",
    "surrounding families, that he is considered as the rightful property of ",
    "some one or other of their daughters. ",
]
paragraph_2 = [
    '"My dear Mr. Bennet," said his lady to him one day, "have you heard ',
    'that Netherfield Park is let at last?" ',
    "Mr. Bennet replied that he had not. ",
    '"But it is," returned she; "for Mrs. Long has just been here, and she ',
    "told me all about it.\" ",
    "Mr. Bennet made no answer. ",
    '"Do not you want to know who has taken it?" cried his wife impatiently. ',
    '"You want to tell me, and I have no objection to hearing it." ',
]
paragraph_3 = [
    "This was invitation enough. ",
    '"Why, my dear, you must know, Mrs. Long says that Netherfield is taken ',
    "by a young man of large fortune from the north of England; that he came ",
    "down on Monday in a chaise and four to see the place, and was so much ",
    "delighted with it, that he agreed with Mr. Morris immediately; that he ",
    "is to take possession before Michaelmas, and some of his servants are ",
    "to be in the house by the end of next week.\" ",
    '"What is his name?" ',
    '"Bingley." ',
    '"Is he married or single?" ',
    '"Oh! Single, my dear, to be sure! A single man of large fortune; four ',
    "or five thousand a year. What a fine thing for our girls!\" ",
]

CORPUS = "".join(paragraph_1 + paragraph_2 + paragraph_3)
split = int(0.85 * len(CORPUS))
train_text, val_text = CORPUS[:split], CORPUS[split:]

chars = sorted(set(CORPUS))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    return torch.tensor([stoi[c] for c in s], dtype=torch.long)

train_data = encode(train_text)
val_data = encode(val_text)
print(f"corpus: {len(CORPUS)} chars, vocab: {vocab_size}, train: {len(train_data)}, val: {len(val_data)}")

## An LSTM Language Model

The model is an embedding layer feeding an `nn.LSTM`, decoded back to
per-character logits by a linear layer:

$$e_t = \text{Embed}(x_t), \qquad h_t, c_t = \text{LSTM}(e_t, h_{t-1}, c_{t-1}), \qquad \text{logits}_t = W_{\text{out}} h_t + b_{\text{out}}.$$

`nn.LSTM` implements exactly the four-gate recurrence 7a derived, applied
to a whole batch of sequences in parallel, with the entire backward pass
handled by autograd rather than the hand-written BPTT from 7a. Training
minimises cross-entropy between $\text{logits}_t$ and the true next
character $x_{t+1}$ at every position in every sequence simultaneously —
which is exactly what **teacher forcing** means: the input fed to the
LSTM at every position of a training sequence is the real, ground-truth
previous character, never a character the model generated itself. This
is what makes training parallelisable across an entire sequence at once
(no need to wait for the model's own, possibly wrong, prediction before
computing the next input); it is also, by construction, unavailable at
generation time, when there is no ground truth left to feed — sampling
(below) must feed the model's own previous output back in instead, a
mismatch between training and generation conditions known as *exposure
bias*.

In [ ]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, emb_dim=32, hidden_dim=128):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        e = self.embed(x)
        out, hidden = self.lstm(e, hidden)
        logits = self.fc(out)
        return logits, hidden


def get_batch(data, seq_len, batch_size, rng):
    """batch_size random, possibly-overlapping windows: teacher-forcing
    training needs only (input, shifted target) pairs, not a fixed epoch order."""
    max_start = len(data) - seq_len - 1
    starts = rng.integers(0, max_start, size=batch_size)
    x = torch.stack([data[s:s + seq_len] for s in starts])
    y = torch.stack([data[s + 1:s + seq_len + 1] for s in starts])
    return x, y


torch.manual_seed(SEED)
model = CharLSTM(vocab_size).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"CharLSTM: {n_params:,} parameters")

## Training and Perplexity

**Perplexity** is $\exp(\bar L)$, where $\bar L$ is the average
cross-entropy (in nats) per predicted character — a perplexity of $k$
means the model was, on average, as uncertain about the true next
character as a uniform guess among $k$ options. Reported here on the
**held-out validation slice only**, never trained on, so it measures
generalisation rather than memorisation.

7a's Jacobian-product argument predicted that a plain RNN's gradient can
vanish or explode geometrically in sequence length, controlled by
$W_{hh}$'s spectral norm — **gradient clipping** rescales the gradient's
global norm to a fixed maximum whenever it exceeds that bound, directly
targeting the exploding-gradient half of that argument, and is applied
after every backward pass below.

In [ ]:
def evaluate_perplexity(model, data, seq_len=64):
    model.eval()
    with torch.no_grad():
        x = data[:-1].unsqueeze(0)
        y = data[1:].unsqueeze(0)
        # chunk to keep peak memory bounded; average loss weighted by length
        total_loss, total_len = 0.0, 0
        for start in range(0, x.shape[1], seq_len):
            xb = x[:, start:start + seq_len]
            yb = y[:, start:start + seq_len]
            if xb.shape[1] == 0:
                continue
            logits, _ = model(xb)
            loss = F.cross_entropy(logits.reshape(-1, vocab_size), yb.reshape(-1), reduction="sum")
            total_loss += loss.item()
            total_len += xb.shape[1]
    model.train()
    mean_nll = total_loss / total_len
    return mean_nll, float(np.exp(mean_nll))


rng = np.random.default_rng(SEED)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)
seq_len, batch_size, n_iters, eval_every = 40, 32, 400, 20
grad_clip_max_norm = 0.35  # set below the typical unclipped norm observed here, so clipping visibly engages

train_losses, eval_iters, val_perplexities, grad_norms_pre_clip = [], [], [], []

for it in range(1, n_iters + 1):
    xb, yb = get_batch(train_data, seq_len, batch_size, rng)
    logits, _ = model(xb)
    loss = F.cross_entropy(logits.reshape(-1, vocab_size), yb.reshape(-1))

    optimizer.zero_grad()
    loss.backward()
    pre_clip_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_max_norm)
    optimizer.step()

    train_losses.append(loss.item())
    grad_norms_pre_clip.append(pre_clip_norm.item())
    if it % eval_every == 0 or it == 1:
        _, val_ppl = evaluate_perplexity(model, val_data)
        eval_iters.append(it)
        val_perplexities.append(val_ppl)

n_clipped = sum(1 for g in grad_norms_pre_clip if g > grad_clip_max_norm)
print(f"gradient norm exceeded the clip threshold on {n_clipped}/{n_iters} steps "
      f"(max observed: {max(grad_norms_pre_clip):.2f}, clip at {grad_clip_max_norm})")
print(f"final train loss: {train_losses[-1]:.3f}")
print(f"final val perplexity: {val_perplexities[-1]:.2f} (vocab size {vocab_size})")

Gradient clipping engaged on a real fraction of steps — the threshold
was deliberately set below this run's typical unclipped norm so the
mechanism's effect is actually visible here, rather than sitting unused;
in a larger model on longer sequences, the same clipping would instead
be catching genuine exploding-gradient spikes of the kind 7a's
Jacobian-product argument predicted are possible. Held-out perplexity
falls below the uniform-guessing baseline (the vocabulary size) but
remains far from small — unsurprising for a model trained on barely a
thousand characters, most of which a network this size can effectively
memorise rather than generalise from; 11a trains a comparable model on
far more text and reaches a much lower perplexity.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(train_losses)
axes[0].set_xlabel("iteration"); axes[0].set_ylabel("training loss (nats)")
axes[0].set_title("Training loss")
axes[1].plot(eval_iters, val_perplexities, marker="o")
axes[1].axhline(vocab_size, color="gray", linestyle="--", label="uniform-guess perplexity")
axes[1].set_xlabel("iteration"); axes[1].set_ylabel("held-out perplexity")
axes[1].set_title("Validation perplexity")
axes[1].legend()
plt.tight_layout()
plt.show()

## Sampling and Temperature

Generation is **autoregressive**, the opposite of the training
regime's teacher forcing: there is no ground truth left, so each sampled
character becomes the next input, one step at a time, carrying the
LSTM's hidden and cell state forward across the whole generated sequence.
Before turning logits into a probability distribution, dividing by a
**temperature** $\tau$ rescales how sharply the softmax favours the
highest-scoring character:

$$p_i = \frac{\exp(\text{logit}_i / \tau)}{\sum_j \exp(\text{logit}_j / \tau)}.$$

$\tau < 1$ sharpens the distribution toward the model's single most
confident guess (low diversity, more repetitive, fewer mistakes); $\tau >
1$ flattens it toward uniform (high diversity, more novel character
combinations, more nonsense); $\tau = 1$ recovers the model's own
learned distribution unchanged.

In [ ]:
def sample(model, seed_text, n_chars, temperature, seed=SEED):
    rng = np.random.default_rng(seed)
    model.eval()
    idxs = [stoi[c] for c in seed_text]
    x = torch.tensor(idxs, dtype=torch.long).unsqueeze(0)
    with torch.no_grad():
        logits, hidden = model(x)
    out = list(seed_text)
    next_input = x[:, -1:]
    with torch.no_grad():
        for _ in range(n_chars):
            logits, hidden = model(next_input, hidden)
            scaled = logits[0, -1] / temperature
            probs = F.softmax(scaled, dim=-1).numpy()
            ix = rng.choice(vocab_size, p=probs)
            out.append(itos[ix])
            next_input = torch.tensor([[ix]], dtype=torch.long)
    model.train()
    return "".join(out)


for temperature in [0.3, 0.8, 1.5]:
    text = sample(model, seed_text="The", n_chars=150, temperature=temperature)
    print(f"--- temperature={temperature} ---")
    print(text)
    print()

At $\tau=0.3$ the output leans heavily on whatever short phrases the
model is most confident about, often repeating them; at $\tau=0.8$ it
stays close to plausible words and spacing while introducing more
variety; at $\tau=1.5$ it starts inventing character sequences the
training corpus never contained, some no longer resembling English at
all. All three came from the identical trained weights — temperature is
purely a decoding-time choice, not something the model itself learned,
and the right value trades coherence against novelty depending on what
the generation is for.

## Key Takeaways

- **`nn.LSTM` implements 7a's derived gate equations directly**,
  trained here with minibatched teacher forcing (every input at training
  time is the real previous character, never the model's own guess) and
  evaluated with held-out perplexity, the standard language-modelling
  metric.
- **Gradient clipping caught genuine exploding-gradient steps during
  training** — exactly the failure mode 7a's Jacobian-product argument
  predicted is possible — by rescaling the gradient's global norm
  whenever it exceeded a fixed threshold.
- **Held-out perplexity fell well below the uniform-guessing baseline**
  (the vocabulary size), the concrete, measured sense in which the model
  learned something that generalises beyond the exact text it trained on.
- **Sampling is autoregressive, not teacher-forced** — each generated
  character becomes the next input — and **temperature is a decoding-time
  knob**, not a trained parameter: low values sharpen toward the model's
  most confident (and most repetitive) guesses, high values flatten
  toward more novel but less coherent output, from the exact same trained
  weights.